# Train a custom wake word: "hey byte"

Adapted from the official [openWakeWord automatic training notebook](https://github.com/dscripka/openWakeWord/blob/main/notebooks/automatic_model_training.ipynb).

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> Run all (about 1 to 1.5 hours; the downloads are large).
At the end, `hey_byte.onnx` downloads to your computer. Put it in `voice_agent/models/` and set `WAKE_WORD` to its full path in `voice_agent/.env`.

Changes from the official demo: more clips and training steps (the demo is deliberately tiny), "bite"-like phrases added as negatives, default accuracy/recall targets kept, and the tflite step dropped (the agent uses ONNX).

**Known risk:** the official setup pins old libraries (TensorFlow 2.8, speechbrain 0.5.14, ...). Colab's Python may be too new for some of them. If an install cell fails, see the notes at the bottom.

## 0. Check the GPU

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU - switch the runtime to T4 GPU")

## 1. Settings (edit these)

In [ ]:
# Longer, more distinctive phrases train much better than one short word. "byte" alone will false-trigger a lot.
TARGET_PHRASE = "hey byte"

# Words that sound similar and must NOT wake it. Includes "bite" because of the jaw-bite project.
NEGATIVE_PHRASES = ["bite", "hey bite", "hey bike", "hey kite", "hey right", "bytes", "hey bright"]

N_SAMPLES = 10000        # synthetic positive clips (demo uses 1000)
N_SAMPLES_VAL = 2000
STEPS = 25000            # training steps (demo uses 10000)

## 2. Environment setup (about 5 to 10 min)

In [ ]:
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize
!pip install webrtcvad

!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword

!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install tensorflow-cpu==2.8.1
!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19

import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

In [ ]:
import os, sys
import numpy as np
import torch
from pathlib import Path
import yaml
import datasets
import scipy
from tqdm import tqdm

## 3. Download training data (large; about 20 GB, 15 to 30 min)

In [ ]:
# Room impulse responses (MIT)
output_dir = "./mit_rirs"
os.makedirs(output_dir, exist_ok=True)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
# Background noise: one AudioSet part + 2 hours of Free Music Archive
os.makedirs("audioset", exist_ok=True)
fname = "bal_train09.tar"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O audioset/{fname} {link}
!cd audioset && tar -xf bal_train09.tar

output_dir = "./audioset_16k"
os.makedirs(output_dir, exist_ok=True)
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

output_dir = "./fma"
os.makedirs(output_dir, exist_ok=True)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))
n_hours = 2
for i in tqdm(range(n_hours*3600//30)):
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
# Precomputed negative-speech features (~2,000 hours) + validation set for false-positive estimates
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

## 4. Training config

In [ ]:
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)

config["target_phrase"] = [TARGET_PHRASE]
config["model_name"] = TARGET_PHRASE.replace(" ", "_")
config["custom_negative_phrases"] = NEGATIVE_PHRASES
config["n_samples"] = N_SAMPLES
config["n_samples_val"] = N_SAMPLES_VAL
config["steps"] = STEPS
# target_accuracy / target_recall / max false-positive rate stay at the file's defaults

config["background_paths"] = ['./audioset_16k', './fma']
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as f:
    yaml.dump(config, f)
print({k: config[k] for k in ("target_phrase", "model_name", "custom_negative_phrases", "n_samples", "steps")})

## 5. Train (3 steps; re-run a step if it stops early)

In [ ]:
# Step 1: synthesize clips with text-to-speech (re-running continues where it left off)
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

In [ ]:
# Step 2: augment with noise and room reverb
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

In [ ]:
# Step 3: train and export. A tflite conversion error at the end is harmless: we only need the .onnx
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

## 6. Download the model

In [ ]:
from google.colab import files
path = f"my_custom_model/{config['model_name']}.onnx"
print(path, os.path.getsize(path), "bytes")
files.download(path)

## After downloading

1. Copy `hey_byte.onnx` to `voice_agent/models/hey_byte.onnx`.
2. In `voice_agent/.env` add: `WAKE_WORD=C:\Users\User\projects\makeuc\voice_agent\models\hey_byte.onnx`
3. Run `python agent.py --test-wake`, say "hey byte" about 10 times, then talk normally and say "bite", "hey right", etc. Tune `WAKE_THRESHOLD` (raise it if it fires by mistake, lower it if it misses you).

## If something fails

- **An install cell errors on a pinned version** (usually tensorflow-cpu 2.8.1 or speechbrain): the official repo's pins are old. Check the [openWakeWord issues](https://github.com/dscripka/openWakeWord/issues) for the current Colab fix. Only the `.onnx` export matters, so tensorflow problems that happen *after* training can be ignored.
- **Generated clips fail midway:** re-run the same cell; it resumes.
- **Out of disk or RAM:** lower `N_SAMPLES` to 5000, or skip the FMA download.
- **Model fires too easily or misses you:** add more `NEGATIVE_PHRASES`, raise `STEPS`, or retrain with `TARGET_PHRASE = "okay byte"`.